# Adoption analysis

Analyses synthetic e-bike adoption patterns across years and population segments. The notebook expects generated conditional synthetic panel files plus feature lists. It produces aggregate adoption, mileage, and segment-level summaries and plots.

# Notebook for CVAE spp analysis

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)

In [ ]:
path = '../'
pop_size = 4
model = 'best_cvae_model'
var = 100

# Load data generated with CVAE - only conditional data here
df_samples_con = pd.read_parquet(path + f"data\TUmodeldata_panel_ally_{pop_size}.parquet")  # Load as DataFrame

# Load feature lists
with open(path +"data/pref_cat.txt", "r") as f:
    cat_features = f.read().splitlines()
with open(path +"data/pref_num.txt", "r") as f:
    num_features = f.read().splitlines()
with open(path +"data/con_num.txt", "r") as f:
    num_cond_features = f.read().splitlines()
with open(path +"data/con_cat.txt", "r") as f:
    cat_cond_features = f.read().splitlines()

In [ ]:
def read_parquet_file_pandas(file, df_samples_con):
    df = pd.read_parquet(file)  # Read as a pandas DataFrame

    # Calculate mileage per mode
    trcol = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
    lencol = ['lengths_mode_1', 'lengths_mode_2', 'lengths_mode_3', 'lengths_mode_4', 'lengths_mode_5', 'lengths_mode_6', 'lengths_mode_7']
    for i in range(len(trcol)):
        mileage_column = f'mileage_mode_{i+1}'  # Create a new column name for mileage
        df[mileage_column] = df[trcol[i]] * df[lencol[i]]

    # Reset the index to ensure known divisions
    df = df.reset_index(drop=True)

    # Reset the index for both DataFrames to ensure known divisions
    df = df.reset_index()
    df_samples_con = df_samples_con.reset_index()

    # Concatenate the two DataFrames
    df_combined = pd.concat([df, df_samples_con], axis=1)

    return df_combined

def concatenate_parquet_files(path, df_samples_con, maxfiles=10):

    # Set the path to the directory containing your Parquet files
    directory_path = path + f"results/samples_{pop_size}_{model}/"

    # List all .parquet files in the directory
    parquet_files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if f.endswith(".parquet")]

    # Limit to the first 100 files
    parquet_files = parquet_files[:maxfiles]

    # Read and concatenate all Parquet files, adding a column for the sample number
    dataframes = []
    for i, file in enumerate(parquet_files):
        # ddf_new = read_parquet_file(file, df_samples_con)
        ddf_new = read_parquet_file_pandas(file, df_samples_con)

        ##### do something

        dataframes.append(ddf_new)
        print(f"Processed file {i+1}/{len(parquet_files)}: {file}")

    return dataframes

In [ ]:
# TU with weights
df = pd.read_csv(path + "data\TUmodeldata_full.csv")

In [ ]:
# Add the Diary year from df to df_samples_con
df['diary_year_fix'] = df['DiaryYear']
df_samples_con = df_samples_con.merge(df[['SessionId', 'diary_year_fix']], on='SessionId', how='left')

In [ ]:
# Weigth the TU with the SessionWeight
dfag_weighted = df.copy()
trip_cols = [
    'trips_mode1', 'trips_mode2', 'trips_mode3',
    'trips_mode4', 'trips_mode5', 'trips_mode6', 'trips_mode7'
]

dfag_weighted = df.copy()

dfag_weighted[trip_cols] = dfag_weighted[trip_cols].multiply(
    dfag_weighted['SessionWeight'], axis=0
)

In [ ]:
# List of trips columns and average length columns
trips_columns = cat_features[1:]
avg_length_columns = num_features

# Calculate mileage for each mode
for i in range(len(trips_columns)):
    mileage_column = f'mileage_mode{i+1}'  # Create a new column name for mileage
    dfag_weighted[mileage_column] = dfag_weighted[trips_columns[i]] * dfag_weighted[avg_length_columns[i]]

## Age and Gender Segments

### Create Segments

In [ ]:
# Map bins back to original datasets
df_samples_con['RespAgeCorrect_q'] = pd.qcut(df_samples_con['RespAgeCorrect'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
dfag_weighted['RespAgeCorrect_q'] = pd.qcut(dfag_weighted['RespAgeCorrect'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# === Define Personas ===
def make_persona(row):
    return f"{int(row['RespSex'])}_Age{row['RespAgeCorrect_q']}"

df_samples_con['persona'] = df_samples_con[['RespAgeCorrect_q','RespSex' ]].apply(make_persona, axis=1)
dfag_weighted['persona'] = dfag_weighted[['RespAgeCorrect_q','RespSex' ]].apply(make_persona, axis=1)

In [ ]:
# Drop menan_lean_mode and trips_mode columns
df_samples_con_join = df_samples_con.drop(columns=avg_length_columns + trips_columns)

# Call the function to read and concatenate Parquet files
ddf_combined = concatenate_parquet_files(path, df_samples_con_join, maxfiles=100)

In [ ]:
dfag = ddf_combined[0]
dfag = dfag[dfag['DiaryYear'] == 2024]

In [ ]:
result = np.array(dfag[['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 
               'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']].sum()) / \
         np.array(dfag[['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 
               'trips_mode_5', 'trips_mode_6', 'trips_mode_7']].sum())

In [ ]:
# Number of trips with average length < 3.4 km for mode 4 (car)
dfag['3_4_car_trips'] = dfag.apply(lambda row: row['trips_mode_4'] if row['lengths_mode_4'] < 3.4 else 0, axis=1)
dfag['4_4_car_trips'] = dfag.apply(lambda row: row['trips_mode_4'] if row['lengths_mode_4'] < 4.4 else 0, axis=1)

# Calculate persentage of trips with average length < 3.4 km for mode 4 (car)
ct_34 = dfag['3_4_car_trips'].sum() / dfag['trips_mode_4'].sum()
ct_44 = dfag['4_4_car_trips'].sum() / dfag['trips_mode_4'].sum()

print(f"Percentage of car trips with average length < 3.4 km: {ct_34:.2%}")
print(f"Percentage of car trips with average length < 4.4 km: {ct_44:.2%}")

### Mode shares development

In [ ]:
def calc_devmet_tu(df, columns_to_sum, segment=['DiaryYear', 'persona']):
    """
    Aggregate TU data with SessionWeight and compute modal-share (%).
    """

    # Aggregate
    df_agg = (
        df.groupby(segment)[columns_to_sum]
        .sum()
        .reset_index()
    )

    # Row-wise normalization (modal share %)
    row_totals = df_agg[columns_to_sum].sum(axis=1)
    df_agg[columns_to_sum] = df_agg[columns_to_sum].div(row_totals, axis=0) * 100

    return df_agg

def calc_devmet(path, df_samples_con, columns_to_sum, segment=['DiaryYear', 'persona'], SPY = None , maxfiles=1001):
    """
    Aggregate CVAE sample files into mean/std modal-split percentages.
    SessionWeight is applied before aggregation to match bootstrap_tu's
    population weighting. Missing segment combinations per file get NaN
    so the cross-file mean is unbiased.
    """
    directory_path = os.path.join(path, f"results/samples_{pop_size}_{model}/")

    parquet_files = [
        os.path.join(directory_path, f)
        for f in os.listdir(directory_path)
        if f.endswith(".parquet")
    ][:maxfiles]

    # Full set of segment keys (from the conditional panel)
    all_segment_keys = (
        df_samples_con[segment]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    purpose_map = {
        0: 'Leisure/Other',
        1: 'Leisure/Other',
        2: 'Leisure/Other',
        3: 'Leisure/Other',
        4: 'Work/Education/Home',
        5: 'Leisure/Other',
        6: 'Work/Education/Home',
        7: 'Leisure/Other',
    }

    dataframes = []
    for i, file in enumerate(parquet_files):
        ddf_new = read_parquet_file_pandas(file, df_samples_con)
        ddf_new['purposes'] = ddf_new['purposes'].map(purpose_map)

        if SPY is not None:
            ddf_new = ddf_new[ddf_new['diary_year_fix'] == SPY]

        ddf_agg = (
            ddf_new.groupby(segment)[columns_to_sum]
            .sum()
            .reset_index()
        )

        # Reindex to full segment keys â†’ missing combos become NaN
        ddf_agg = all_segment_keys.merge(ddf_agg, on=segment, how='left')

        # Row-wise normalisation to modal-split percentages
        row_totals = ddf_agg[columns_to_sum].sum(axis=1)
        ddf_agg[columns_to_sum] = ddf_agg[columns_to_sum].div(row_totals, axis=0) * 100

        dataframes.append(ddf_agg)

        if (i + 1) % 50 == 0:
            print(f"  Processing file {i + 1}/{len(parquet_files)}...")

    df = pd.concat(dataframes)
    df = df.groupby(segment).agg(['mean', 'std']).reset_index()
    df.columns = ['_'.join(col).strip('_') for col in df.columns.values]

    return df

In [ ]:
columns_to_sum = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
ddf_combined = calc_devmet(path, df_samples_con[['DiaryYear']], columns_to_sum, maxfiles=var, segment=['DiaryYear'])
# Remove dash before number in column names
ddf_combined.columns = [col.replace('trips_mode_', 'trips_mode') for col in ddf_combined.columns]
ebike_cvae_trips_ave = ddf_combined[['DiaryYear','trips_mode3_std','trips_mode3_mean']]

In [ ]:
# Calculate the average persentage increase per year
ebike_cvae_trips_ave['perc_increase'] = ebike_cvae_trips_ave['trips_mode3_mean'].pct_change() * 100
ebike_cvae_trips_ave

In [ ]:
# Calculate weighted CVAE with devmet for mileage
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil = calc_devmet(path, df_samples_con[['DiaryYear']], columns_to_sum, maxfiles=var, segment=['DiaryYear'])
ddf_combined_mil.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil.columns]
ebike_cvae_mil_aver = ddf_combined_mil[['DiaryYear', 'mileage_mode3_std','mileage_mode3_mean']]

In [ ]:
# Calculate the average persentage increase per year
ebike_cvae_mil_aver['perc_increase'] = ebike_cvae_mil_aver['mileage_mode3_mean'].pct_change() * 100
ebike_cvae_mil_aver

### Segments trips

In [ ]:
def read_parquet_file_pandas(file, df_samples_con):
    df = pd.read_parquet(file)  # Read as a pandas DataFrame

    # Calculate mileage per mode
    trcol = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
    lencol = ['lengths_mode_1', 'lengths_mode_2', 'lengths_mode_3', 'lengths_mode_4', 'lengths_mode_5', 'lengths_mode_6', 'lengths_mode_7']
    for i in range(len(trcol)):
        mileage_column = f'mileage_mode_{i+1}'  # Create a new column name for mileage
        df[mileage_column] = df[trcol[i]] * df[lencol[i]]

    # Reset the index to ensure known divisions
    df = df.reset_index(drop=True)

    # Reset the index for both DataFrames to ensure known divisions
    df = df.reset_index()
    df_samples_con = df_samples_con.reset_index()

    # Concatenate the two DataFrames
    df_combined = pd.concat([df, df_samples_con], axis=1)

    return df_combined

In [ ]:
# Calculate weighted TU with bootstrapping for trips
columns_to_sum = ['trips_mode1', 'trips_mode2', 'trips_mode3', 'trips_mode4', 'trips_mode5', 'trips_mode6', 'trips_mode7']
ebike_tu_trips = calc_devmet_tu(dfag_weighted, columns_to_sum)
ebike_tu_trips = ebike_tu_trips[['DiaryYear', 'persona','trips_mode3']]

In [ ]:
def apply_moving_average(df, value_col, window=3):
    df_smooth = df.copy()
    
    df_smooth[value_col] = (
        df_smooth
        .sort_values(['persona', 'DiaryYear'])
        .groupby('persona')[value_col]
        .transform(lambda x: x.rolling(window=window, center=True, min_periods=1).mean())
    )
    
    return df_smooth

# Create smoothed TU
ebike_tu_trips_smooth = apply_moving_average(ebike_tu_trips, 'trips_mode3', window=3)

In [ ]:
# Calculate CVAE with devmet for trips
columns_to_sum = ['trips_mode_1', 'trips_mode_2', 'trips_mode_3', 'trips_mode_4', 'trips_mode_5', 'trips_mode_6', 'trips_mode_7']
ddf_combined = calc_devmet(path, df_samples_con[['DiaryYear', 'persona']], columns_to_sum, maxfiles=var)
# Remove dash before number in column names
ddf_combined.columns = [col.replace('trips_mode_', 'trips_mode') for col in ddf_combined.columns]
ebike_cvae_trips = ddf_combined[['DiaryYear', 'persona','trips_mode3_mean']]

In [ ]:
def compute_volatility(df, value_col):
    vals = []

    for persona in df['persona'].unique():
        subset = df[df['persona'] == persona].sort_values('DiaryYear')
        diff = subset[value_col].diff().dropna()

        if len(diff) > 0:
            vals.append(diff.var())

    return np.mean(vals)

def compute_rank_stability(df, value_col):
    years = sorted(df['DiaryYear'].unique())
    corrs = []

    for t in range(len(years) - 1):
        y1, y2 = years[t], years[t+1]

        d1 = df[df['DiaryYear'] == y1].set_index('persona')[value_col]
        d2 = df[df['DiaryYear'] == y2].set_index('persona')[value_col]

        common = d1.index.intersection(d2.index)

        if len(common) > 1:
            c, _ = spearmanr(d1.loc[common], d2.loc[common])
            corrs.append(c)

    return np.nanmean(corrs)


def compute_reversals(df, value_col):
    vals = []

    for persona in df['persona'].unique():
        subset = df[df['persona'] == persona].sort_values('DiaryYear')

        diff = subset[value_col].diff().dropna()
        diff = diff[diff != 0]

        signs = np.sign(diff).values  # âœ… FIX: convert to array

        if len(signs) > 1:
            reversals = np.sum(signs[1:] * signs[:-1] < 0)
            vals.append(reversals)
        else:
            vals.append(0)

    return np.mean(vals)

In [ ]:
results = []

# TU raw
results.append({
    'method': 'TU_raw',
    'volatility': compute_volatility(ebike_tu_trips, 'trips_mode3'),
    'rank_corr': compute_rank_stability(ebike_tu_trips, 'trips_mode3'),
    'reversals': compute_reversals(ebike_tu_trips, 'trips_mode3')
})

# TU smoothed
results.append({
    'method': 'TU_smooth',
    'volatility': compute_volatility(ebike_tu_trips_smooth, 'trips_mode3'),
    'rank_corr': compute_rank_stability(ebike_tu_trips_smooth, 'trips_mode3'),
    'reversals': compute_reversals(ebike_tu_trips_smooth, 'trips_mode3')
})

# CVAE
results.append({
    'method': 'CVAE',
    'volatility': compute_volatility(ebike_cvae_trips, 'trips_mode3_mean'),
    'rank_corr': compute_rank_stability(ebike_cvae_trips, 'trips_mode3_mean'),
    'reversals': compute_reversals(ebike_cvae_trips, 'trips_mode3_mean')
})

results_df = pd.DataFrame(results).round(4)
print(results_df)

In [ ]:
# --- Define consistent color palette for colorblind accessibility ---
persona_order = ebike_tu_trips['persona'].unique()  # Get unique personas for consistent color mapping
persona_order = sorted(persona_order)  # Ensure the same order in both subplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Helper: moving average
def moving_average(series, window=3):
    return series.rolling(window=window, center=True, min_periods=1).mean()

# Helper: Plot a line for each persona
def plot_lines_share(ax, data, col, title, smooth=False):
    for persona in persona_order:
        subset = data[data['persona'] == persona].sort_values('DiaryYear')

        y = subset[col]

        # Apply smoothing only if requested
        if smooth:
            y = moving_average(y, window=3)

        ax.plot(
            subset['DiaryYear'],
            y,
            label=persona,
            marker='o',
        )

    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.grid(True)

# --- Plot CVAE (no variance, no smoothing) ---
plot_lines_share(axes[0], ebike_cvae_trips, 'trips_mode3_mean', 'CVAE', smooth=False)
axes[0].set_ylabel('(%)')
axes[0].legend().remove()

# --- Plot TU (with moving average smoothing) ---
plot_lines_share(axes[1], ebike_tu_trips, 'trips_mode3', 'Moving Average', smooth=True)
axes[1].set_ylabel('')
axes[1].legend().remove()

# --- Shared Legend at Bottom ---
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=8, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.ylim(0, 5)
plt.savefig(path + "results/plots/ebike_share_trips_cvae_tu.png", bbox_inches='tight', dpi=500)
plt.show()

### Mileage development

In [ ]:
# Calculate weighted TU with bootstrapping for mileage
columns_to_sum = ['mileage_mode1', 'mileage_mode2', 'mileage_mode3', 'mileage_mode4', 'mileage_mode5', 'mileage_mode6', 'mileage_mode7']
ebike_tu_mil = calc_devmet_tu(dfag_weighted, columns_to_sum)
ebike_tu_mil = ebike_tu_mil[['DiaryYear', 'persona','mileage_mode3']]

In [ ]:
# Create smoothed TU
ebike_tu_mil_smooth = apply_moving_average(ebike_tu_mil, 'mileage_mode3', window=3)

In [ ]:
# Calculate weighted CVAE with devmet for mileage
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil = calc_devmet(path, df_samples_con[['DiaryYear', 'persona']], columns_to_sum, maxfiles=var)
ddf_combined_mil.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil.columns]
ebike_cvae_mil = ddf_combined_mil[['DiaryYear', 'persona','mileage_mode3_mean']]

In [ ]:
results = []

# TU raw
results.append({
    'method': 'TU_raw',
    'volatility': compute_volatility(ebike_tu_mil, 'mileage_mode3'),
    'rank_corr': compute_rank_stability(ebike_tu_mil, 'mileage_mode3'),
    'reversals': compute_reversals(ebike_tu_mil, 'mileage_mode3')
})

# TU smoothed
results.append({
    'method': 'TU_smooth',
    'volatility': compute_volatility(ebike_tu_mil_smooth, 'mileage_mode3'),
    'rank_corr': compute_rank_stability(ebike_tu_mil_smooth, 'mileage_mode3'),
    'reversals': compute_reversals(ebike_tu_mil_smooth, 'mileage_mode3')
})

# CVAE
results.append({
    'method': 'CVAE',
    'volatility': compute_volatility(ebike_cvae_mil, 'mileage_mode3_mean'),
    'rank_corr': compute_rank_stability(ebike_cvae_mil, 'mileage_mode3_mean'),
    'reversals': compute_reversals(ebike_cvae_mil, 'mileage_mode3_mean')
})

results_df = pd.DataFrame(results).round(4)
print(results_df)

In [ ]:
# --- Create subplots ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# --- Plot CVAE (raw) ---
plot_lines_share(axes[0], ebike_cvae_mil, 'mileage_mode3_mean', 'CVAE', smooth=False)
axes[0].set_ylabel('(%)')
axes[0].legend().remove()
# 
# --- Plot TU (smoothed) ---
plot_lines_share(axes[1], ebike_tu_mil, 'mileage_mode3', 'Moving Average', smooth=True)
axes[1].set_ylabel('')
axes[1].legend().remove()

# --- Shared Legend at Bottom ---
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=8, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.ylim(0, 2)

plt.savefig(path + "results/plots/ebike_share_mileage_cvae_tu.png",
            bbox_inches='tight', dpi=500)

plt.show()


## Spatial distribution of e bike adoption

In [ ]:
df_samples_con['HomeAdrGMMzoneL1'] = df_samples_con['HomeAdrGMMzoneL2'] - df_samples_con['HomeAdrGMMzoneL2'] % 100
df_samples_con['HomeAdrGMMzoneL0'] = df_samples_con['HomeAdrGMMzoneL2'] - df_samples_con['HomeAdrGMMzoneL2'] % 1000

In [ ]:
# Calculate weighted CVAE with devmet for mileage
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con[['DiaryYear', 'HomeAdrGMMzoneL1']], columns_to_sum, maxfiles=var, segment = ['DiaryYear', 'HomeAdrGMMzoneL1'])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_mil2 = ddf_combined_mil2[['DiaryYear', 'HomeAdrGMMzoneL1','mileage_mode3_std','mileage_mode3_mean']]

In [ ]:
import geopandas as gpd

# Path to the GDB file
gdb_file = './data/in_Zones_GMM40_2.gdb'

# Read the GDB file
gdf = gpd.read_file(gdb_file)

# Genarate zone levels
gdf['zoneidL2'] = gdf['zoneid'] - gdf['zoneid'] % 10
gdf['zoneidL1'] = gdf['zoneid'] - gdf['zoneid'] % 100

In [ ]:
merged_gdf = gdf.dissolve(by='zoneidL1', aggfunc={
    'komnr': 'first',
    'zonenavn': 'first',
    'zonedescrip': 'first',
    'numadresser': 'sum',
    'Korrigeret': 'max',
    'Shape_Length': 'sum',
    'Shape_Area': 'sum'
}).reset_index()

# --- Join the two dataframes ---
joined_df = merged_gdf.merge(ebike_cvae_mil2, right_on='HomeAdrGMMzoneL1', left_on='zoneidL1', how='left')
joined_df['DiaryYear'] = pd.to_datetime(joined_df['DiaryYear'], format='%Y').dt.year

In [ ]:
# Filter for 2015 and 2024
df_2015 = joined_df[joined_df['DiaryYear'] == 2015][['geometry', 'zoneidL1', 'DiaryYear',  'mileage_mode3_mean']].rename(columns={'mileage_mode3_mean': 'share_pct_2015'})
df_2024 = joined_df[joined_df['DiaryYear'] == 2024][['geometry', 'zoneidL1', 'DiaryYear', 'mileage_mode3_mean']].rename(columns={'mileage_mode3_mean': 'share_pct_2024'})

In [ ]:
# Merge on zoneidL1
merged_gdf2 = df_2015.merge(df_2024[[ 'zoneidL1', 'DiaryYear', 'share_pct_2024']], on='zoneidL1')

# Calculate percentage change
merged_gdf2['pct_change_15_24'] = ((merged_gdf2['share_pct_2024'] - merged_gdf2['share_pct_2015']) / merged_gdf2['share_pct_2015']) * 100

In [ ]:
# Save the GeoDataFrame to a file
output_file = path + 'results/ebike_adoption_zones.geojson'  # Specify your desired output file path and format
joined_df[['zoneidL1', 'geometry', 'Shape_Length', 'Shape_Area', 'DiaryYear', 'mileage_mode3_std', 'mileage_mode3_mean']].to_file(output_file, driver='GeoJSON')

df_2015.to_file(path + 'results/ebike_adoption_zones15.geojson', driver='GeoJSON')
df_2024.to_file(path + 'results/ebike_adoption_zones24.geojson', driver='GeoJSON')
merged_gdf2.to_file(path + 'results/ebike_adoption_zonesdiff.geojson', driver='GeoJSON')

## Detailed segments

In [ ]:
def plot_grouped_trends(
    data, 
    group_col, 
    title="Grouped Trend", 
    y_col_mean="mileage_mode3_mean", 
    y_col_std="mileage_mode3_std", 
    x_col="DiaryYear",
    ylabel="(%)", 
    figsize=(12, 5),
    sharey=True,
    show_legend=True
):
    
    # Create subplot
    fig, ax = plt.subplots(1, 1, figsize=figsize, sharey=sharey)

    # Plot each group
    for persona in data[group_col].unique():
        subset = data[data[group_col] == persona].sort_values(by=x_col)
        ax.plot(
            subset[x_col],
            subset[y_col_mean],
            label=persona,
            marker='o',
        )

        # Add shaded std area if available
        if y_col_std in subset.columns:
            ax.fill_between(
                subset[x_col],
                subset[y_col_mean] - subset[y_col_std],
                subset[y_col_mean] + subset[y_col_std],
                alpha=0.2,
            )

    # Format plot
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.grid(True)
    if show_legend:
        ax.legend(title=group_col)

    plt.tight_layout()
    plt.show()

    return fig, ax

#### Gender

In [ ]:
col_to_group = 'RespSex'
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear', col_to_group])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_trips_aveG = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode3_std','mileage_mode3_mean']]

In [ ]:
print(ebike_cvae_trips_aveG[ebike_cvae_trips_aveG['DiaryYear'] == 2015])
print(ebike_cvae_trips_aveG[ebike_cvae_trips_aveG['DiaryYear'] == 2024])
plot_grouped_trends(
    ebike_cvae_trips_aveG,
    group_col=col_to_group,
    title='CVAE'
)

#### Age

In [ ]:
# Create quartiles
col_to_cut = 'RespAgeCorrect'
quantiles = df_samples_con[col_to_cut].quantile([0.25, 0.5, 0.75])
print(quantiles)
bins= [df_samples_con[col_to_cut].min()-1, quantiles[0.25], quantiles[0.5], quantiles[0.75], df_samples_con[col_to_cut].max()+1]
labels = ['Q1', 'Q2', 'Q3', 'Q4']
df_samples_con[col_to_cut + 'Group'] = pd.cut(df_samples_con[col_to_cut], bins=bins, labels=labels) 

# Calcluate milage by quartiles
col_to_group = col_to_cut + 'Group'
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear', col_to_group])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_trips_aveA = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode3_std','mileage_mode3_mean']]

print(ebike_cvae_trips_aveA[ebike_cvae_trips_aveA['DiaryYear'] == 2015])
print(ebike_cvae_trips_aveA[ebike_cvae_trips_aveA['DiaryYear'] == 2024])
plot_grouped_trends(
    ebike_cvae_trips_aveA,
    group_col=col_to_group,
    title='CVAE'
)

#### Income

In [ ]:
# Create quartiles
quantiles = df_samples_con['IncFamily2023'].quantile([0.25, 0.5, 0.75])
print(quantiles)
bins= [df_samples_con['IncFamily2023'].min()-1, quantiles[0.25], quantiles[0.5], quantiles[0.75], df_samples_con['IncFamily2023'].max()+1]
labels = ['Q1', 'Q2', 'Q3', 'Q4']
df_samples_con['IncomeGroup'] = pd.cut(df_samples_con['IncFamily2023'], bins=bins, labels=labels) 

# Calcluate milage by quartiles
col_to_group = 'IncomeGroup'
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear', col_to_group])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_trips_aveI = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode3_std','mileage_mode3_mean']]

print(ebike_cvae_trips_aveI[ebike_cvae_trips_aveI['DiaryYear'] == 2015])
print(ebike_cvae_trips_aveI[ebike_cvae_trips_aveI['DiaryYear'] == 2024])
plot_grouped_trends(
    ebike_cvae_trips_aveI,
    group_col=col_to_group,
    title='CVAE'
)

#### Car accessibility pop

In [ ]:
# Function to assign quartiles within each year
def assign_quartiles(group):
    # Compute year-specific quartiles
    q = group['access_pop_exp_car'].quantile([0.25, 0.5, 0.75])
    
    # Define bins using these quartiles
    bins = [
        group['access_pop_exp_car'].min() - 1,
        q.loc[0.25],
        q.loc[0.5],
        q.loc[0.75],
        group['access_pop_exp_car'].max() + 1
    ]
    
    # Assign quartile labels
    group['access_pop_exp_carGroup'] = pd.cut(
        group['access_pop_exp_car'], 
        bins=bins, 
        labels=['Q1', 'Q2', 'Q3', 'Q4'],
        include_lowest=True
    )
    
    return group

# Apply per-year
df_samples_con = df_samples_con.groupby('DiaryYear', group_keys=False).apply(assign_quartiles)

In [ ]:
col_to_group = 'access_pop_exp_carGroup'
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear', col_to_group])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_trips_aveCAP = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode3_std','mileage_mode3_mean']]

In [ ]:
print(ebike_cvae_trips_aveCAP[ebike_cvae_trips_aveCAP['DiaryYear'] == 2015])
print(ebike_cvae_trips_aveCAP[ebike_cvae_trips_aveCAP['DiaryYear'] == 2024])
plot_grouped_trends(
    ebike_cvae_trips_aveCAP,
    group_col=col_to_group,
    title='CVAE'
)

#### Car ownership

In [ ]:
# If more than 3 car trips make it 3
df_samples_con['HousehNumcarsv2'] = df_samples_con['HousehNumcars'].apply(lambda x: 3 if x > 3 else x)

In [ ]:
col_to_group = 'HousehNumcarsv2'
columns_to_sum = ['mileage_mode_1', 'mileage_mode_2', 'mileage_mode_3', 'mileage_mode_4', 'mileage_mode_5', 'mileage_mode_6', 'mileage_mode_7']
ddf_combined_mil2 = calc_devmet(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear', col_to_group])
ddf_combined_mil2.columns = [col.replace('mileage_mode_', 'mileage_mode') for col in ddf_combined_mil2.columns]
ebike_cvae_trips_aveCO = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode3_std','mileage_mode3_mean']]

print(ebike_cvae_trips_aveCO[ebike_cvae_trips_aveCO['DiaryYear'] == 2015])
print(ebike_cvae_trips_aveCO[ebike_cvae_trips_aveCO['DiaryYear'] == 2024])
plot_grouped_trends(
    ebike_cvae_trips_aveCO,
    group_col=col_to_group,
    title='CVAE'
)

### Purpose

In [ ]:
import os
import pandas as pd

def calc_devmet_purp(
    path,
    df_samples_con,
    columns_to_sum,
    segment=['DiaryYear', 'purposes'],
    maxfiles=1001
):
    """
    Aggregate CVAE sample files into mean/std modal-split percentages,
    including purpose segmentation.
    """

    directory_path = os.path.join(path, f"results/samples_{pop_size}_{model}/")

    parquet_files = [
        os.path.join(directory_path, f)
        for f in os.listdir(directory_path)
        if f.endswith(".parquet")
    ][:maxfiles]

    # --- Purpose mapping ---
    purpose_map = {
        0: 'Leisure/Other',
        4: 'Work/Education/Home',
        6: 'Work/Education/Home',
        2: 'Leisure/Other',
        3: 'Leisure/Other',
        1: 'Leisure/Other',
        7: 'Leisure/Other',
        5: 'Leisure/Other'
    }

    # --- Build full segment grid (IMPORTANT FIX) ---
    base_keys = (
        df_samples_con[['DiaryYear']]
        .drop_duplicates()
    )

    purpose_categories = pd.DataFrame({
        'purposes': ['Work/Education/Home', 'Leisure/Other']
    })

    all_segment_keys = (
        base_keys.assign(key=1)
        .merge(purpose_categories.assign(key=1), on='key')
        .drop('key', axis=1)
    )

    # --- Loop over files ---
    dataframes = []

    for i, file in enumerate(parquet_files):
        ddf_new = read_parquet_file_pandas(file, df_samples_con)

        # Map purposes
        ddf_new['purposes'] = ddf_new['purposes'].map(purpose_map)

        # Aggregate
        ddf_agg = (
            ddf_new.groupby(segment)[columns_to_sum]
            .sum()
            .reset_index()
        )

        # Ensure full grid (missing combos = NaN)
        ddf_agg = all_segment_keys.merge(ddf_agg, on=segment, how='left')

        # Row-wise normalization (modal split %)
        row_totals = ddf_agg[columns_to_sum].sum(axis=1)
        ddf_agg[columns_to_sum] = (
            ddf_agg[columns_to_sum]
            .div(row_totals, axis=0) * 100
        )

        dataframes.append(ddf_agg)

        if (i + 1) % 50 == 0:
            print(f"Processing file {i + 1}/{len(parquet_files)}...")

    # --- Combine all files ---
    df = pd.concat(dataframes)

    # Mean + std across samples
    df = (
        df.groupby(segment)[columns_to_sum]   # âœ… only numeric columns
        .agg(['mean', 'std'])
        .reset_index()
    )

    # Flatten column names
    df.columns = ['_'.join(col).strip('_') for col in df.columns.values]

    return df

In [ ]:
col_to_group = 'purposes'
ddf_combined_mil2 = calc_devmet_purp(path, df_samples_con, columns_to_sum, maxfiles=var, segment = ['DiaryYear','purposes'])
ebike_cvae_trips_aveP = ddf_combined_mil2[['DiaryYear', col_to_group,'mileage_mode_3_std','mileage_mode_3_mean']]

print(ebike_cvae_trips_aveP[ebike_cvae_trips_aveP['DiaryYear'] == 2024])
print(ebike_cvae_trips_aveP[ebike_cvae_trips_aveP['DiaryYear'] == 2015])